# 02. 모델 · 입력 크기 비교 학습

여러 조합을 **짧게(20바퀴)** 학습해서 비교하고, 제일 좋은 조합만 **길게(60바퀴)** 다시 학습해요.

| 순서 | MODEL | IMGSZ | EPOCHS | 목적 | 예상 시간(T4) |
| --- | --- | --- | --- | --- | --- |
| 1 | yolov8n | 960 | 20 | 기준 | 약 45분 |
| 2 | yolov8n | 640 | 20 | 입력 크기 비교 | 약 20분 |
| 3 | yolov8n | 1280 | 20 | 입력 크기 비교 | 약 1시간 30분 |
| 4 | yolov8s | 960 | 20 | 모델 크기 비교 | 약 1시간 30분 |
| 5 | yolo11n | 960 | 20 | 모델 세대 비교 | 약 50분 |
| + | yolo26n | 960 | 60 | 최신 모델 비교 (NMS-free 구조, 엣지용) | 약 2시간 30분 |
| 6 | (1~5 중 최고) | (최고) | 60 | 최종 베이스라인 | 2~4시간 |
| 7 | yolo11n | 1152 | 90 | **본선용**: 증류(선생 v8s) + 클래스 가중치 | 약 6~7시간 |
| 8 | (7번 결과) | | 10 | **QAT** (8번 칸) | 약 1시간 |
| 9 | yolo11m | 1280 | 60 | **95점 도전 1단계**: 새 선생 (train+dev) | 약 30시간 이상 |
| 10 | yolo11n | 1280 | 100 | **95점 도전 2단계**: 증류(선생 11m) + 클래스 가중치 1.0 + train+dev | 약 15시간 이상 |
| 11 | (10번 결과) | | 10 | **95점 도전 3단계**: QAT (8번 칸) | 약 2시간 |
| 12 | yolo11n | 1152 | 90 | **사람 점수 올리기**: 7번과 같은 설정 + DontCare 가리기 + 사람 사진 2번 (`PED = True`) | 약 8시간 |
| 13 | (12번 결과) | | 10 | QAT (8번 칸) | 약 1시간 |

- **5번 칸의 `MODEL`, `IMGSZ`, `EPOCHS` 세 줄만** 바꿔서 한 번씩 실행하면 돼요.
- 비교는 **dev 셋(쪽지시험)** 점수로만 해요. 시험지(eval)로 고르면 커닝이에요.
- 결과는 **Google Drive**에 저장돼요. 접속이 끊겨도 같은 설정으로 다시 실행하면 이어서 학습해요.
- 무료 Colab은 하루 GPU 사용 시간이 정해져 있어서, 하루에 2~3개씩 나눠서 하는 게 안전해요.
- 시간은 대략적인 값이에요.
- `CUDA out of memory` 에러가 나면: 그때까지 한 바퀴들은 저장돼 있어요. 세션 다시 시작 → 1~4번 → 5번을 같은 설정으로 다시 실행하면 이어서 해요.

## 1. GPU 확인

`Tesla T4` 가 나오면 OK. 에러가 나면 런타임 → 런타임 유형 변경 → **T4 GPU** 로 바꿔 주세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive 연결

권한 허용 창이 뜨면 허용해 주세요. 학습 결과가 `내 드라이브/embeded_ondeviceAI/runs` 에 저장돼요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS = '/content/drive/MyDrive/embeded_ondeviceAI/runs'
os.makedirs(RUNS, exist_ok=True)

## 3. 데이터 준비

01 노트북과 같은 작업이에요. 이번 접속에서 01을 이미 실행했다면 금방 끝나요.

In [ ]:
%%bash
set -e   # 중간에 실패하면 바로 멈추고 어디서 실패했는지 보여 줌
git clone -q https://github.com/gyu-young-04/embeded_ondeviceAI.git /content/embeded_ondeviceAI 2>/dev/null || git -C /content/embeded_ondeviceAI pull -q
mkdir -p /content/kitti && cd /content/kitti
for f in data_object_label_2.zip data_object_image_2.zip; do
  wget -c -q https://s3.eu-central-1.amazonaws.com/avg-kitti/$f
  unzip -q -n $f 'training/*'
done
[ "$(ls training/image_2 | wc -l)" = 7481 ] || { echo '사진이 7481장이 아니에요. /content/kitti 를 지우고 다시 실행하세요.'; exit 1; }
cd /content/embeded_ondeviceAI
[ -f /content/kitti_yolo/kitti.yaml ] || python scripts/prepare_kitti.py --kitti /content/kitti --out /content/kitti_yolo
# train + dev 를 합쳐서 학습하는 설정 파일 (FULL = True 일 때 사용). eval 은 여전히 빠져 있어요
[ -f /content/kitti_yolo/kitti_full.yaml ] || sed 's#^train: train.txt#train: [train.txt, dev.txt]#' /content/kitti_yolo/kitti.yaml > /content/kitti_yolo/kitti_full.yaml
grep -q 'train: \[train.txt, dev.txt\]' /content/kitti_yolo/kitti_full.yaml || { echo 'kitti_full.yaml 만들기 실패'; exit 1; }
# 사람 점수용 변형 데이터 (PED = True 일 때 사용): DontCare 가리기 + 사람 사진 2번. 처음 한 번 몇 분 걸려요
[ -f /content/kitti_yolo_dc/kitti_dc.yaml ] || python scripts/make_ped_data.py --kitti /content/kitti --src /content/kitti_yolo --out /content/kitti_yolo_dc
wc -l /content/kitti_yolo/train.txt /content/kitti_yolo/dev.txt /content/kitti_yolo_dc/train.txt

## 4. Ultralytics 설치

출력되는 버전을 보고서의 '실험 환경'에 적어 두세요.

In [ ]:
!pip -q install ultralytics==8.4.189 nvidia-modelopt==0.48.0   # 증류·QAT 가 들어 있는 버전으로 고정. modelopt 는 03 과 같은 버전이어야 QAT 모델을 읽을 수 있어요
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)

## 5. 학습 ⭐ 여기 세 줄만 바꿔서 실행

- 위 표의 순서대로 `MODEL`, `IMGSZ`, `EPOCHS` 를 바꾸고 ▶ 를 눌러요.
- 끊기면 1~4번을 다시 실행한 뒤 이 칸을 **같은 설정으로** 실행하면 이어서 학습해요.
- 화면에 나오는 mAP50 / mAP50-95 는 COCO 방식이라 **대회 점수(KITTI AP40)와 달라요.** 비교용으로만 봐요.
- 7번 실험부터는 `TEACHER`(증류 선생), `CLS_PW`(클래스 가중치)도 써요. 예전 실험을 다시 돌릴 땐 `TEACHER = None`, `CLS_PW = 0` 으로 두면 예전과 같아요.
- 9번 실험부터는 `FULL = True` 로 **dev 도 학습에 넣어요** (데이터 11% 증가). 이때 화면의 dev 점수는 이미 본 문제라 **부풀려져서** 예전 실험과 비교하면 안 돼요. 진짜 점수는 03 노트북(eval 1,000장)으로만 봐요.

- 12번 실험은 `PED = True` 로 **사람 점수를 올리는 데이터**를 써요. 7번과 다른 건 데이터뿐이라 점수 차이 = 데이터 효과예요.
  - **DontCare 가리기**: 라벨이 없는 '너무 먼 물체' 영역을 회색으로 칠해요. 안 그러면 모델이 그 안의 작은 사람을 '사람 아님'으로 배워요.
  - **사람 사진 2번**: 사람이 있는 사진을 한 바퀴에 두 번 봐요 (예시가 적은 사람을 더 많이 공부).
  - 바뀌는 건 train 사진뿐이고, dev·eval 은 원본 그대로예요. dev 점수는 7번과 **그대로 비교해도 돼요.**

**사람 점수 올리기 설정 (12번)**

| MODEL | IMGSZ | EPOCHS | TEACHER | CLS_PW | FULL | PED | 실험 이름 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `'yolo11n'` | 1152 | 90 | `'yolov8s_960_e60'` | 0.5 | `False` | `True` | `yolo11n_1152_e90_kd_cw_ped` |

**95점 도전 설정 (순서대로)**

| 단계 | MODEL | IMGSZ | EPOCHS | TEACHER | CLS_PW | FULL | 실험 이름 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1. 선생 | `'yolo11m'` | 1280 | 60 | `None` | 0.5 | `True` | `yolo11m_1280_e60_cw_full` |
| 2. 학생 | `'yolo11n'` | 1280 | 100 | `'yolo11m_1280_e60_cw_full'` | 1.0 | `True` | `yolo11n_1280_e100_kd_cw1.0_full` |


In [ ]:
MODEL = 'yolo11n'   # yolov8n / yolov8s / yolo11n / yolo26n
IMGSZ = 1152        # 640 / 960 / 1152 / 1280
EPOCHS = 90         # 비교할 때 20, 최종 학습은 60, 본선용 90
TEACHER = 'yolov8s_960_e60'   # 증류 학습 선생 (runs 폴더 안 실험 이름). 안 쓰면 None
CLS_PW = 0.5        # 클래스 가중치: 0 = 끔, 0.5 = 수가 적은 사람·자전거를 틀리면 벌점을 더 크게, 1.0 = 더 강하게
FULL = False        # True = dev 도 학습에 넣기 (95점 도전용)
PED = False         # True = DontCare 가리기 + 사람 사진 2번 (사람 점수 올리기)

from pathlib import Path
import torch
from ultralytics import YOLO

cw = '' if not CLS_PW else '_cw' if CLS_PW == 0.5 else f'_cw{CLS_PW}'   # 0.5 는 예전 이름 그대로
assert not (FULL and PED), 'FULL 과 PED 는 아직 같이 못 써요. 하나만 True 로 두세요.'
NAME = f'{MODEL}_{IMGSZ}_e{EPOCHS}' + ('_kd' if TEACHER else '') + cw + ('_full' if FULL else '') + ('_ped' if PED else '')
if FULL:
    DATA = '/content/kitti_yolo/kitti_full.yaml'
elif PED:
    DATA = '/content/kitti_yolo_dc/kitti_dc.yaml'
else:
    DATA = '/content/kitti_yolo/kitti.yaml'
# GPU 메모리(15GB)에 맞춘 한 번에 보는 사진 수
if MODEL.endswith('m'):
    BATCH = 4                      # 중간 모델(m)은 무거워요
elif TEACHER and IMGSZ >= 1280:
    BATCH = 8                      # 1280 + 선생도 같이 돌아감
elif IMGSZ >= 1152 or MODEL.endswith('s') or TEACHER:
    BATCH = 16
else:
    BATCH = 32
extra = {}
if TEACHER:
    teacher_pt = Path(RUNS) / TEACHER / 'weights' / 'best.pt'
    assert teacher_pt.exists(), f'선생 모델이 없어요: {teacher_pt}'
    extra['distill_model'] = str(teacher_pt)
print('실험 이름:', NAME, '| batch', BATCH, '| 데이터', DATA)
last = Path(RUNS) / NAME / 'weights' / 'last.pt'

def finished(ckpt):
    # Ultralytics 는 학습이 끝나면 체크포인트의 epoch 를 -1 로 바꿔 둔다
    return torch.load(ckpt, map_location='cpu', weights_only=False).get('epoch', 0) == -1

if last.exists() and finished(last):
    print('이미 끝난 실험이에요:', NAME)
elif last.exists():
    print('이어서 학습:', last)
    YOLO(str(last)).train(resume=True, batch=BATCH)
else:
    YOLO(f'{MODEL}.pt').train(
        data=DATA,
        imgsz=IMGSZ,
        epochs=EPOCHS,
        patience=20,      # 20바퀴 동안 나아지지 않으면 일찍 멈춤
        batch=BATCH,
        cache='ram',      # 메모리가 부족하면 자동으로 꺼져요
        cls_pw=CLS_PW,
        seed=0,
        deterministic=True,
        project=RUNS,
        name=NAME,
        exist_ok=True,
        **extra,
    )

## 5-2. 지금 실험 방식 학습 ⭐ (Kaggle 노트북과 같은 설정 · 같은 실험 이름)

Kaggle 과 동시에 다른 실험을 돌릴 때 써요. 1~4번을 실행한 뒤 이 칸만 실행해요 (5번 칸은 건너뛰어요).

- 설정 이름과 뜻은 Kaggle 노트북 1번 칸과 같아요. 실험 이름도 똑같이 만들어져서 Kaggle 결과와 섞어 비교할 수 있어요.
- 결과는 Drive `runs/실험이름/` 에 바로 저장돼요. 끊기면 1~4번 → 이 칸을 **같은 설정으로** 다시 실행하면 이어서 해요.
- 선생 가중치는 Drive `runs/선생이름/weights/best.pt` 에 있어야 해요.

In [ ]:
MODEL = 'yolo11n'
IMGSZ = 1280
EPOCHS = 90
TEACHER = 'yolo11n_1152_e90_kd_cw_ped'   # Drive runs 안의 선생 실험 이름. 안 쓰면 None
KD_TAG = ''          # 선생을 바꾸면 숫자를 넣어요 (이름이 _kd2 처럼 바뀜)
CLS_PW = 0.5
FULL = False         # True = dev 도 학습에 넣기 (마지막 재학습 때만)
PED = True
PED_DUP = 3          # 사람 사진을 한 바퀴에 몇 번 볼지
PED_HARD = False     # True = 어려운 사람 사진만 PED_DUP 번, 나머지 사람 사진은 1번
PED_CROP = False     # True = 어려운 사람 확대 사진 추가
SCALE = 0.9          # 크기 증강 (기본 0.5)
BOX = 10             # 박스 손실 가중치 (기본 7.5)
CLS = 0.5            # 분류 손실 가중치 (기본 0.5)
SEED = 0

import re, subprocess
from pathlib import Path
import torch
from ultralytics import YOLO

cw = '' if not CLS_PW else '_cw' if CLS_PW == 0.5 else f'_cw{CLS_PW}'
ped_tag = f'{PED_DUP}' + ('h' if PED_HARD else '') + ('c' if PED_CROP else '')
ped = '' if not PED else '_ped' if ped_tag == '2' else f'_ped{ped_tag}'
NAME = (f'{MODEL}_{IMGSZ}_e{EPOCHS}' + (f'_kd{KD_TAG}' if TEACHER else '') + cw + ('_full' if FULL else '') + ped
        + ('' if SCALE == 0.5 else f'_s{SCALE}') + ('' if BOX == 7.5 else f'_box{BOX}')
        + ('' if CLS == 0.5 else f'_cls{CLS}') + (f'_seed{SEED}' if SEED else ''))

print('실험 이름:', NAME)
if PED:   # 사람 데이터: 설정마다 따로 만들어요 (처음 한 번 5~10분)
    tag = ped_tag + ('f' if FULL else '')
    out = '/content/kitti_yolo_dc' if tag == '2' else f'/content/kitti_yolo_dc{tag}'
    args = ['--dup', '1', '--hard-dup', str(PED_DUP)] if PED_HARD else ['--dup', str(PED_DUP)]
    args += (['--crop', '1'] if PED_CROP else []) + (['--full'] if FULL else [])
    if not Path(out, 'kitti_dc.yaml').exists():
        print('사람 데이터 만드는 중 (5~10분, 끝나면 결과가 한꺼번에 나와요)...')
        r = subprocess.run(['python', 'scripts/make_ped_data.py', '--kitti', '/content/kitti', '--src', '/content/kitti_yolo',
                            '--out', out, *args], cwd='/content/embeded_ondeviceAI', capture_output=True, text=True)
        print(r.stdout, r.stderr[-3000:] if r.returncode else '')
        r.check_returncode()
    DATA = f'{out}/kitti_dc.yaml'
elif FULL:
    DATA = '/content/kitti_yolo/kitti_full.yaml'
else:
    DATA = '/content/kitti_yolo/kitti.yaml'

if MODEL.endswith('m'):
    BATCH = 4
elif TEACHER and IMGSZ >= 1280:
    BATCH = 8
elif IMGSZ >= 1152 or MODEL.endswith('s') or TEACHER:
    BATCH = 16
else:
    BATCH = 32
extra = {}
if TEACHER:
    teacher_pt = Path(RUNS) / TEACHER / 'weights' / 'best.pt'
    assert teacher_pt.exists(), f'선생 모델이 없어요: {teacher_pt}'
    extra['distill_model'] = str(teacher_pt)
print('실험 이름:', NAME, '| batch', BATCH, '| 데이터', DATA)

last = Path(RUNS) / NAME / 'weights' / 'last.pt'
done = last.exists() and torch.load(last, map_location='cpu', weights_only=False).get('epoch', 0) == -1
if done:
    print('이미 끝난 실험이에요:', NAME)
elif last.exists():
    print('이어서 학습:', last)
    YOLO(str(last)).train(resume=True, batch=BATCH)
else:
    YOLO(f'{MODEL}.pt').train(data=DATA, imgsz=IMGSZ, epochs=EPOCHS, patience=20, batch=BATCH, cache='ram',
                              cls_pw=CLS_PW, scale=SCALE, box=BOX, cls=CLS, seed=SEED, deterministic=True,
                              project=RUNS, name=NAME, exist_ok=True, **extra)

## 6. 실험 비교표

Drive에 저장된 모든 실험을 표로 모아요. **`dev mAP50-95` 가 높을수록 정확**해요.
표는 `runs/comparison.csv` 로도 저장돼요. 속도(latency) 비교는 다음 노트북에서 따로 재요.

In [ ]:
import pandas as pd
import torch
from pathlib import Path

rows = []
for run in sorted(Path(RUNS).iterdir()):
    csv = run / 'results.csv'
    if not csv.exists():
        continue
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    best = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
    last = run / 'weights' / 'last.pt'
    done = last.exists() and torch.load(last, map_location='cpu', weights_only=False).get('epoch', 0) == -1
    rows.append({
        '실험': run.name,
        '진행 바퀴': len(df),
        'dev mAP50': round(best['metrics/mAP50(B)'], 3),
        'dev mAP50-95': round(best['metrics/mAP50-95(B)'], 3),
        '학습 시간(분)': round(df['time'].iloc[-1] / 60) if 'time' in df else None,
        '상태': '완료' if done else '진행 중',
    })

table = pd.DataFrame(rows).sort_values('dev mAP50-95', ascending=False)
table.to_csv(Path(RUNS) / 'comparison.csv', index=False)
table

## 7. 클래스별 점수 보기 (선택)

아래 `NAME` 에 보고 싶은 실험 이름을 넣으면 Car / Pedestrian / Cyclist 별 점수를 보여 줘요.

In [ ]:
NAME = 'yolov8n_960_e20'

from IPython.display import Image, display
run = Path(RUNS) / NAME
display(Image(str(run / 'results.png'), width=1000))
imgsz = int(NAME.split('_')[1])
metrics = YOLO(str(run / 'weights' / 'best.pt')).val(data='/content/kitti_yolo/kitti.yaml', imgsz=imgsz, split='val', batch=16)
for i, n in metrics.names.items():
    print(f'{n:<11} mAP50 {metrics.box.ap50[i]:.3f}  mAP50-95 {metrics.box.ap[i]:.3f}')

## 8. QAT (양자화 인식 학습) — 본선용

다 학습한 모델에서 출발해서 **INT8로 바뀔 때 생기는 오차를 흉내 내며** 10바퀴 더 학습해요.
모델이 그 오차에 미리 적응해서, INT8로 바꿔도 점수가 덜 떨어져요.

- `BASE` 에 출발할 실험 이름을 넣어요 (7번 실험, 95점 도전이면 `yolo11n_1280_e100_kd_cw1.0_full`).
- 데이터(train+dev 여부)와 클래스 가중치는 `BASE` 이름을 보고 학습 때와 똑같이 맞춰요.
- 이미 잘 학습된 모델을 살짝 고치는 거라 **학습률을 작게**(0.001), 처음 몸풀기(warmup)는 끄고, 모자이크도 꺼요.
- 끝나면 03 노트북 **15번 칸**에서 INT8 파일로 바꾸고 채점해요.
- 끊기면 1~4번 → 이 칸을 다시 실행하면 이어서 해요.


In [ ]:
BASE = 'yolo11n_1152_e90_kd_cw'   # QAT 출발점
QAT_EPOCHS = 10

from pathlib import Path
from ultralytics import YOLO
base_pt = Path(RUNS) / BASE / 'weights' / 'best.pt'
assert base_pt.exists(), f'{base_pt} 가 없어요. 5번 칸 학습이 끝났는지 확인하세요.'
QNAME = f'{BASE}_qat{QAT_EPOCHS}'
imgsz = int(BASE.split('_')[1])
if BASE.endswith('_full'):
    QDATA = '/content/kitti_yolo/kitti_full.yaml'
elif BASE.endswith('_ped'):
    QDATA = '/content/kitti_yolo_dc/kitti_dc.yaml'
else:
    QDATA = '/content/kitti_yolo/kitti.yaml'
QCLS_PW = 1.0 if '_cw1.0' in BASE else 0.5 if '_cw' in BASE else 0
QBATCH = 8 if imgsz >= 1280 else 16
print('QAT 이름:', QNAME, '| 데이터', QDATA, '| cls_pw', QCLS_PW, '| batch', QBATCH)
last = Path(RUNS) / QNAME / 'weights' / 'last.pt'

if last.exists() and finished(last):
    print('이미 끝난 실험이에요:', QNAME)
elif last.exists():
    print('이어서 학습:', last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(str(base_pt)).train(
        data=QDATA,
        imgsz=imgsz,
        epochs=QAT_EPOCHS,
        quantize=8,              # ← QAT 켜기 (NVIDIA ModelOpt 가 자동 설치돼요)
        optimizer='SGD',
        lr0=0.001,               # 처음 학습(0.01)보다 10배 작게: 살짝만 고침
        warmup_epochs=0,
        close_mosaic=QAT_EPOCHS, # 모자이크 끔
        cls_pw=QCLS_PW,
        batch=QBATCH,
        cache='ram',
        seed=0,
        deterministic=True,
        project=RUNS,
        name=QNAME,
        exist_ok=True,
    )


## 8-2. 지금 방식 QAT ⭐ (Kaggle 노트북 QAT 와 같은 설정 · 같은 실험 이름)

1~4번을 실행한 뒤 이 칸만 실행해요 (5, 5-2, 8번 칸은 건너뛰어요).

- `BASE` 이름을 보고 데이터(사람 사진 몇 번, 확대 여부), 크기 증강, box/cls 를 학습 때와 똑같이 맞춰요.
- `QAT_KD = True` 면 출발 모델(FP32)을 선생으로 써서, INT8 흉내를 내는 동안에도 점수가 FP32 처럼 유지되게 해요.
- 출발 모델은 Drive `runs/BASE/weights/best.pt` 에 있어야 해요.
- QAT 는 이어하기가 안 돼요(Ultralytics 버그). 끊기면 1~4번 → 이 칸을 다시 실행하면 **처음부터** 다시 해요.
- 끝나면 03 노트북 15번 칸에서 `QAT_TAG = '_qkd'` 로 채점해요.

In [ ]:
BASE = 'yolo11n_1280_e90_kd_cw_ped3_s0.9'   # QAT 출발 실험 이름 (Drive runs 안)
QAT_EPOCHS = 10
QAT_SCALE = None    # None = 출발 모델과 같은 크기 증강. 숫자를 넣으면 이름에 _qs 가 붙어요
QAT_LR = 0.001      # 바꾸면 이름에 _qlr 이 붙어요
QAT_KD = True       # True = 출발 모델(FP32)을 선생으로 증류. 이름에 _qkd 가 붙어요

import re, shutil, subprocess
from pathlib import Path
import torch
from ultralytics import YOLO

base_pt = Path(RUNS) / BASE / 'weights' / 'best.pt'
assert base_pt.exists(), f'출발 모델이 없어요: {base_pt}'
QAT_NAME = (f'{BASE}_qat{QAT_EPOCHS}' + ('' if QAT_SCALE is None else f'_qs{QAT_SCALE}')
            + ('' if QAT_LR == 0.001 else f'_qlr{QAT_LR}') + ('_qkd' if QAT_KD else ''))
imgsz = int(BASE.split('_')[1])

m = re.search(r'_ped(\d*h?c?)', BASE)   # 학습 때 데이터와 똑같이
if m:
    tag = (m.group(1) or '2') + ('f' if '_full' in BASE else '')
    out = '/content/kitti_yolo_dc' if tag == '2' else f'/content/kitti_yolo_dc{tag}'
    n = re.match(r'\d+', tag).group()
    args = ['--dup', '1', '--hard-dup', n] if 'h' in tag else ['--dup', n]
    args += (['--crop', '1'] if 'c' in tag else []) + (['--full'] if 'f' in tag else [])
    if not Path(out, 'kitti_dc.yaml').exists():
        print('사람 데이터 만드는 중 (몇 분)...', out)
        r = subprocess.run(['python', 'scripts/make_ped_data.py', '--kitti', '/content/kitti', '--src', '/content/kitti_yolo',
                            '--out', out, *args], cwd='/content/embeded_ondeviceAI', capture_output=True, text=True)
        print(r.stdout[-1500:])
        if r.returncode:
            print(r.stderr[-3000:])
        r.check_returncode()
    QDATA = f'{out}/kitti_dc.yaml'
elif '_full' in BASE:
    QDATA = '/content/kitti_yolo/kitti_full.yaml'
else:
    QDATA = '/content/kitti_yolo/kitti.yaml'
ms = re.search(r'_s(\d+\.?\d*)', BASE)
QSCALE = QAT_SCALE if QAT_SCALE is not None else float(ms.group(1)) if ms else 0.5
qloss = {k: float(mm.group(1)) for k in ('box', 'cls') if (mm := re.search(rf'_{k}(\d+\.?\d*)', BASE))}
QCLS_PW = 1.0 if '_cw1.0' in BASE else 0.5 if '_cw' in BASE else 0
QBATCH = 8 if imgsz >= 1280 else 16
qkd = {'distill_model': str(base_pt)} if QAT_KD else {}   # 선생 = 같은 모델의 FP32
print('QAT 출발:', BASE, '→', QAT_NAME, '| 데이터', QDATA, '| scale', QSCALE, '| batch', QBATCH, qloss)

last = Path(RUNS) / QAT_NAME / 'weights' / 'last.pt'
if last.exists() and torch.load(last, map_location='cpu', weights_only=False).get('epoch', 0) == -1:
    print('이미 끝난 실험이에요:', QAT_NAME)
else:
    if (Path(RUNS) / QAT_NAME).exists():
        print('⚠️ QAT 는 이어하기가 안 돼서 처음부터 다시 해요:', QAT_NAME)
        shutil.rmtree(Path(RUNS) / QAT_NAME)
    YOLO(str(base_pt)).train(data=QDATA, imgsz=imgsz, epochs=QAT_EPOCHS, quantize=8, optimizer='SGD', lr0=QAT_LR,
                             warmup_epochs=0, close_mosaic=QAT_EPOCHS, cls_pw=QCLS_PW, scale=QSCALE, batch=QBATCH,
                             cache='ram', seed=0, deterministic=True, project=RUNS, name=QAT_NAME, exist_ok=True,
                             **qloss, **qkd)
    print('✅ 끝났어요:', QAT_NAME)